In [1]:
import pandas as pd

In [2]:
equipamentos = [
    "IJ-044",
    "IJ-046",
    "IJ-117",
    "IJ-118",
    "IJ-119",
    "IJ-120",
    "IJ-121",
    "IJ-122",
    "IJ-123",
    "IJ-124",
    "IJ-125",
    "IJ-129",
    "IJ-130",
    "IJ-131",
    "IJ-132",
    "IJ-133",
    "IJ-134",
    "IJ-135",
    "IJ-136",
    "IJ-137",
    "IJ-138",
    "IJ-139",
    "IJ-151",
    "IJ-152",
    "IJ-155",
    "IJ-156",
    "IJ-164",
]

In [3]:
# Store the dates as a list of strings
dates = [
    "26-May",
    "28-Jan",
    "10-Feb",
    "11-Feb",
    "10-Feb",
    "24-Feb",
    "25-Feb",
    "24-Feb",
    "11-Feb",
    "7-Feb",
    "20-Nov",
    "25-Feb",
    "18-May",
    "25-Feb",
    "18-Feb",
    "2-Mar",
    "2-Mar",
    "2-Mar",
    "18-May",
    "17-Mar",
    "16-Mar",
    "3-Mar",
    "9-Mar",
    "9-Mar",
    "16-Mar",
    "19-May",
    "17-Mar",
]

# Prepend '2024-' to each date string
dates = ["2024-" + date for date in dates]

# Convert the dates to datetime format
dates_datetime = pd.to_datetime(dates, format="%Y-%d-%b")

# Format the dates as "YYYY-MM-DD"
formatted_dates = [date.strftime("%Y-%m-%d") for date in dates_datetime]

# Print the formatted dates
assert len(equipamentos) == len(formatted_dates)

In [4]:
# Create the dictionary with keys from equipamentos and values from formatted_dates
equipamento_dates = dict(zip(equipamentos, formatted_dates))

# Print the dictionary
print(equipamento_dates)

{'IJ-044': '2024-05-26', 'IJ-046': '2024-01-28', 'IJ-117': '2024-02-10', 'IJ-118': '2024-02-11', 'IJ-119': '2024-02-10', 'IJ-120': '2024-02-24', 'IJ-121': '2024-02-25', 'IJ-122': '2024-02-24', 'IJ-123': '2024-02-11', 'IJ-124': '2024-02-07', 'IJ-125': '2024-11-20', 'IJ-129': '2024-02-25', 'IJ-130': '2024-05-18', 'IJ-131': '2024-02-25', 'IJ-132': '2024-02-18', 'IJ-133': '2024-03-02', 'IJ-134': '2024-03-02', 'IJ-135': '2024-03-02', 'IJ-136': '2024-05-18', 'IJ-137': '2024-03-17', 'IJ-138': '2024-03-16', 'IJ-139': '2024-03-03', 'IJ-151': '2024-03-09', 'IJ-152': '2024-03-09', 'IJ-155': '2024-03-16', 'IJ-156': '2024-05-19', 'IJ-164': '2024-03-17'}


In [5]:
for index, value in enumerate(equipamento_dates):
    path = "IJ-118/"
    file_ = value + ".csv"
    data_final_manutencao = str(equipamento_dates[value])
    print(file_, data_final_manutencao)

    pd.set_option("display.max_rows", None)
    pd.set_option("display.max_columns", None)

    # Read the CSV file into a DataFrame
    df = pd.read_csv(path + file_)

    # # Display the first 5 rows
    # print(df.head().to_markdown(index=False, numalign="left", stralign="left"))

    # # Print the column names and their data types
    # print(df.info())

    # Convert `Data de Produção` to datetime
    df["Data de Produção"] = pd.to_datetime(df["Data de Produção"])

    # Calculate the days until '2024-05-26'
    df["Manutencao"] = (
        pd.to_datetime(data_final_manutencao) - df["Data de Produção"]
    ).dt.days

    # Drop the original `Data de Produção` column
    df = df.drop("Data de Produção", axis=1)

    # # Display the first 5 rows
    # print(df.head().to_markdown(index=False, numalign="left", stralign="left"))

    # # Print the column names and their data types
    # print(df.info())
    df.to_csv("mnt-" + file_, index=False)

    # Sort by `Manutencao`
    df.sort_values(by="Manutencao", inplace=True)

    # Group by `Manutencao` and calculate the cumulative sum for the specified columns
    df["Qtd. Produzida"] = df.groupby("Manutencao")["Qtd. Produzida"].cumsum()
    df["Qtd. Refugada"] = df.groupby("Manutencao")["Qtd. Refugada"].cumsum()
    df["Qtd. Retrabalhada"] = df.groupby("Manutencao")["Qtd. Retrabalhada"].cumsum()
    df["Fator Un."] = df.groupby("Manutencao")["Fator Un."].cumsum()
    # Convert `"Consumo de massa no item em (Kg/100pçs)"` to numeric
    # Remove any non-numeric characters from the column
    df["Consumo de massa no item em (Kg/100pçs)"] = (
        df["Consumo de massa no item em (Kg/100pçs)"]
        .astype(str)
        .str.replace("[^0-9.]", "", regex=True)
    )

    # Convert the column to numeric
    df["Consumo de massa no item em (Kg/100pçs)"] = pd.to_numeric(
        df["Consumo de massa no item em (Kg/100pçs)"]
    )

    # Now you can apply cumsum
    df["Consumo de massa no item em (Kg/100pçs)"] = df.groupby("Manutencao")[
        "Consumo de massa no item em (Kg/100pçs)"
    ].cumsum()

    # Group by `Manutencao` and calculate the last values for the specified columns
    df_result = df.groupby("Manutencao").last().reset_index()

    # Display the first 5 rows of the resulting data
    print(df_result.head().to_markdown(index=False, numalign="left", stralign="left"))

    # Print the column names and their data types
    print(df_result.info())

    # Sort the dataframe by `Manutencao` in descending order
    df_sorted = df_result.sort_values(by="Manutencao", ascending=False)

    # Print the first 5 rows of the sorted dataframe
    print(df_sorted.head().to_markdown(index=False, numalign="left", stralign="left"))

    # Group by `Manutencao` and calculate the last values for the specified columns
    df_result = df.groupby("Manutencao").last().reset_index()

    # Reorder columns to place 'Manutencao' last
    cols = df_result.columns.tolist()
    cols.remove("Manutencao")
    cols.append("Manutencao")
    df_result = df_result[cols]

    # # Display the first 5 rows of the resulting data
    # print(df_result.head().to_markdown(index=False, numalign="left", stralign="left"))

    # # Print the column names and their data types
    # print(df_result.info())

    # Sort the dataframe by `Manutencao` in descending order
    df_sorted = df_result.sort_values(by="Manutencao", ascending=False)
    df_result = df_sorted.reset_index(drop=True)

    df_result.to_csv("mnt-grouped-" + file_, index=False)

    # Calculate the cumulative sum for the specified columns and store them in new columns
    df_result["Qtd. Produzida Acumulada Total"] = df_result["Qtd. Produzida"].cumsum()
    df_result["Qtd. Refugada Acumulada Total"] = df_result["Qtd. Refugada"].cumsum()
    df_result["Qtd. Retrabalhada Acumulada Total"] = df_result[
        "Qtd. Retrabalhada"
    ].cumsum()
    df_result["Fator Un. Acumulado Total"] = df_result["Fator Un."].cumsum()
    df_result["Consumo de massa no item em (Kg/100pçs) Acumulado Total"] = df_result[
        "Consumo de massa no item em (Kg/100pçs)"
    ].cumsum()

    # Get the list of columns
    cols = df_result.columns.tolist()

    # Remove 'Manutencao' from the list
    cols.remove("Manutencao")

    # Append 'Manutencao' to the end of the list
    cols.append("Manutencao")

    # Reorder the dataframe with the new column order
    df_result = df_result[cols]
    print(df_result.info())
    df_result.to_csv("mnt-oficial-" + file_, index=False)

IJ-044.csv 2024-05-26
| Manutencao   | Cód. Ordem   | Cód. Recurso   | Cód. Produto   | Qtd. Produzida   | Qtd. Refugada   | Qtd. Retrabalhada   | Fator Un.   | Cód. Un.   | Descrição da massa (Composto)   | Consumo de massa no item em (Kg/100pçs)   |
|:-------------|:-------------|:---------------|:---------------|:-----------------|:----------------|:--------------------|:------------|:-----------|:--------------------------------|:------------------------------------------|
| 0            | 2697563      | IJ-044         | SA07371        | 114              | 4               | 0                   | 1           | PC         | F424/85                         | 7                                         |
| 1            | 2697563      | IJ-044         | SA07371        | 412              | 0               | 0                   | 2           | PC         | F424/85                         | 14                                        |
| 2            | 2697563      | IJ-044         | SA07371  